In [ ]:
# Install hypertools (dev-1.0 preview) -- run this first on Colab.
# On release this becomes: %pip install hypertools
%pip install -q "hypertools[interactive] @ git+https://github.com/ContextLab/hypertools.git@dev-1.0"

%matplotlib inline

# Morphing through the shapes zoo, with titles

One `hyp.plot(..., animate='morph')` call smoothly *morphs* a cloud of black
dots from one shape to the next, holding on each shape before flowing into
the following one. HyperTools ships a "shapes zoo" of classic 3-D point
clouds (`bunny`, `cube`, `sphere`, `vase`, ...), each downloaded once and
then cached in `~/hypertools_data`, so this notebook is fully offline and
deterministic after the first run. On a cold cache with no network it says
so and morphs five parametric stand-ins instead, so it always renders.

The **title that tracks the current shape** comes straight from the
library: passing a list of per-shape names as `title=` is enough. A morph
animation alternates *hold* segments (the camera slowly orbits a finished
shape) with *transition* segments (one shape flowing into the next);
`hyp.plot` names the shape while holding and shows nothing mid-transition,
so the label never sits over a half-formed cloud. Nothing here recomputes
the morph schedule or reaches into private modules.

## 1. Imports

In [1]:
import os
from typing import NamedTuple

import numpy as np

import hypertools as hyp

## 2. The zoo, and how it is assembled

`normalize` is not redundant with `hyp.plot`: the plot rescales every
dataset with one shared affine, so clouds left in their raw units would be
drawn at wildly different sizes. The sampling is done by hand rather than
left to `morph_samples` because the loop-closing repeat of the first cloud
has to be the *same* sample; `morph_samples=N` is still passed so the cap is
explicit. The teapot ships with only 301 unique coordinates out of 1728
rows, so its segment reads sparser than its neighbours; that is the dataset,
not this notebook.

In [2]:
# NOTE on the teapot: ``hyp.load('teapot')`` returns 1728 rows but only 301
# UNIQUE coordinates (ratio 0.174, measured 2026-07-26), where every other shape
# is essentially all-unique (bunny 35947/35947, vase 36022/36022, cube
# 30034/30246, sphere 29891/30135). Its segment therefore draws with a few
# hundred distinct dots rather than a couple of thousand and reads sparser than
# its neighbours. That is the shipped dataset, not a fault in this example.
SHAPES = ['bunny', 'cube', 'sphere', 'teapot', 'vase']
# normalize() below is NOT redundant with hyp.plot: plot rescales every dataset
# with ONE shared affine, so clouds left in their own raw units would be drawn
# at wildly different sizes. The sampling IS done by hand rather than left to
# morph_samples: the loop-closing repeat of the first cloud has to be the SAME
# sample, and morph_samples draws a fresh subset per dataset. The cap itself
# is what keeps the morph tractable: the one-to-one point matching is a
# Hungarian assignment costing roughly O(n^3), and the zoo's clouds have ~30k
# points each. Passing morph_samples=N as well makes the cap explicit and
# reproducible rather than a silent default.
N = 2000
# the normalized cube reaches +/-1 on every axis, i.e. exactly the drawn axes
# box, so its frames read as noise in a wireframe rather than as a cube; shrink
# it to sit visibly inside. The other shapes still set the shared box.
CUBE_SCALE = 0.8


class Shapes(NamedTuple):
    clouds: list                # sampled, normalized, loop-closed
    titles: list                # one per cloud
    source: str                 # which path produced them


def normalize(points):
    """Center a point cloud and scale it into the hypertools [-1, 1] cube."""
    points = np.asarray(points, dtype=float)
    points = points - points.mean(axis=0)
    return points / np.abs(points).max()


def assemble(clouds, n, source, seed=0):
    """Normalize, shrink the cube, sample n points, and close the loop:
    morphing back to the FIRST shape means a looping player never hard-cuts
    from the last shape to the first, and reusing the same sampled array
    means the closing hold and the opening hold draw an identical point set."""
    rng = np.random.default_rng(seed)
    sampled = []
    for name, points in clouds.items():
        points = normalize(points) * (CUBE_SCALE if name == 'cube' else 1.0)
        sampled.append(points[rng.choice(len(points), size=min(n, len(points)),
                                         replace=False)])
    titles = [name.capitalize() for name in clouds]
    return Shapes(sampled + [sampled[0]], titles + [titles[0]], source)

## 3. Load the zoo, with a parametric stand-in

`load_shapes` is the real path; `hyp.load` used to be the one loader in the
gallery that hard-failed offline, so it now degrades to five parametric
clouds and prints why. `fixture_data` builds the same payload from those
clouds, and is what the test-suite drives.

In [3]:
def synthetic_shapes(n=N, seed=0):
    """Five parametric clouds standing in for the zoo when it cannot be
    fetched: the morph is the same technique on any point clouds."""
    rng = np.random.default_rng(seed)
    u, v = rng.uniform(0, 2 * np.pi, n), rng.uniform(-1, 1, n)
    ring = np.sqrt(1 - v ** 2)
    return {
        'sphere': np.column_stack([ring * np.cos(u), ring * np.sin(u), v]),
        'cube': rng.uniform(-1, 1, (n, 3)),
        'torus': np.column_stack([(1 + 0.4 * np.cos(np.pi * v)) * np.cos(u),
                                  (1 + 0.4 * np.cos(np.pi * v)) * np.sin(u),
                                  0.4 * np.sin(np.pi * v)]),
        'helix': np.column_stack([np.cos(3 * np.pi * v), np.sin(3 * np.pi * v), v]),
        'cone': np.column_stack([(1 - v) / 2 * np.cos(u), (1 - v) / 2 * np.sin(u), v]),
    }


def load_shapes(shapes=SHAPES, n=N):
    """The ONLY function here that may touch the network. ``hyp.load`` caches
    the zoo under ``~/hypertools_data``; on a cold cache with no network it
    raises, and this degrades to the parametric stand-ins and says so."""
    if os.environ.get('HYPERTOOLS_OFFLINE'):
        raise RuntimeError('HYPERTOOLS_OFFLINE is set: refusing to fetch')
    try:
        return assemble({name: hyp.load(name) for name in shapes}, n,
                        'the hypertools shapes zoo')
    except Exception as error:
        print(f'shapes zoo unavailable ({error!r}); using parametric stand-ins')
        return assemble(synthetic_shapes(n), n, 'parametric stand-ins (offline)')


def fixture_data():
    """The same payload from the parametric clouds. No network, no bytes."""
    return assemble(synthetic_shapes(), N, 'parametric stand-ins (fixture)')

## 4. One call: morph, with one title per shape

`rotations` gives each segment its screen time (a full turn per hold, half
a turn per transition; the first hold is split across the two ends so the
loop is seamless), and `title=` takes one string per cloud.

In [4]:
def construct_artifact(data):
    """`data.clouds` / `data.titles` in, the animation out. Returns the
    HyperAnimation wrapper, never the unpacked pair."""
    # [hold_1, morph_1->2, hold_2, ..., hold_N]: with the loop-closing copy
    # there are 6 clouds, so 2*6 - 1 = 11 segments. Camera speed is constant,
    # so these ratios set each segment's SCREEN TIME: a full turn per shape,
    # half a turn per transition. The first shape's hold is split in half
    # across the two ends (they play back-to-back on repeat, giving one full
    # hold), and the total, 8.0, is a whole number of turns, so the azimuth
    # also wraps exactly at the loop point.
    rotations = [0.75] + [0.5, 1.0] * (len(data.clouds) - 2) + [0.5, 0.75]
    # THE hypertools call: black pixel-sized dots morphing through the zoo.
    # title= names each shape while its hold plays and is left blank by
    # hyp.plot itself during every transition -- no hand-rolled schedule.
    anim = hyp.plot(data.clouds, fmt='.', color='k', markersize=1.6,
                    animate='morph', rotations=rotations, morph_samples=N,
                    duration=12, frame_rate=20, size=(6, 6), show=False,
                    title=data.titles)
    return anim

## 5. Build the animation

In [5]:
shapes = load_shapes()
print(f'shapes: {len(shapes.clouds) - 1} clouds + the loop-closing copy, '
      f'{N} points each ({shapes.source})')
anim = construct_artifact(shapes)
fig = anim.figure
_ = anim.draw_frame(anim.n_frames // 4)   # a quarter of the way round the zoo

shapes: 5 clouds + the loop-closing copy, 2000 points each (the hypertools shapes zoo)


## 6. Save the animation

In [6]:
anim.save('morph_zoo.gif', dpi=100)
print('saved morph_zoo.gif')

saved morph_zoo.gif


![morphing through the shapes zoo](morph_zoo.gif)